# Module 01: Vectors, Dot Products & What Quantization Destroys

**Time**: ~30 minutes

**Prerequisites**: You know that embeddings represent meaning as vectors, and that similar things have similar embeddings. You have seen cosine similarity used to measure relatedness.

**Goal**: Understand *geometrically* what dot products measure, then see firsthand why naive compression destroys the relationships between vectors -- even when individual values look fine.

---

In [ ]:
import sys; sys.path.insert(0, "..")
from src.helpers import check_intuition, section_header, run_quantize_experiment

import torch
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

torch.manual_seed(42)

## 1. From Embeddings to Dot Products

You already know the punchline: embeddings turn meaning into geometry. Two sentences about dogs end up near each other; a sentence about dogs and one about tax law end up far apart.

But what does "near" actually mean? The workhorse is the **dot product** (and its normalized cousin, **cosine similarity**).

Let's start with something concrete. We will create fake "embedding" vectors and measure their similarity.

In [ ]:
section_header("From Embeddings to Dot Products", "What similarity actually computes")

# Imagine these are 8-dimensional embeddings for three concepts
dog = torch.randn(8)
puppy = dog + 0.3 * torch.randn(8)  # similar to dog
tax_law = torch.randn(8)             # unrelated

# Raw dot products
print(f"dog . puppy   = {torch.dot(dog, puppy).item():.4f}")
print(f"dog . tax_law = {torch.dot(dog, tax_law).item():.4f}")
print()

# Cosine similarity (normalized dot product)
def cosine_sim(a, b):
    return torch.dot(a, b) / (a.norm() * b.norm())

print(f"cosine(dog, puppy)   = {cosine_sim(dog, puppy).item():.4f}")
print(f"cosine(dog, tax_law) = {cosine_sim(dog, tax_law).item():.4f}")
print()
print("Higher dot product / cosine = more similar. Simple as that.")

## 2. What a Dot Product Actually Computes

The dot product has a clean geometric interpretation:

$$\mathbf{a} \cdot \mathbf{b} = \|\mathbf{a}\| \, \|\mathbf{b}\| \, \cos(\theta)$$

where $\theta$ is the angle between the two vectors. Equivalently:

$$\cos(\theta) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \, \|\mathbf{b}\|}$$

This means:
- **Cosine = 1**: vectors point in the same direction (identical meaning)
- **Cosine = 0**: vectors are orthogonal (unrelated)
- **Cosine = -1**: vectors point in opposite directions

There is also the **projection** interpretation: the dot product $\mathbf{a} \cdot \mathbf{b}$ tells you how much of $\mathbf{a}$ lies along the direction of $\mathbf{b}$, scaled by $\|\mathbf{b}\|$.

Let's see this in 2D first.

In [ ]:
section_header("2D Visualization", "Dot product = projection")

# Two 2D vectors
a = torch.tensor([3.0, 1.0])
b = torch.tensor([1.0, 2.0])

# Projection of a onto b
b_unit = b / b.norm()
proj_scalar = torch.dot(a, b_unit)  # how far along b does a reach?
proj_vector = proj_scalar * b_unit   # the actual projection vector

dot_val = torch.dot(a, b).item()
cos_val = cosine_sim(a, b).item()
angle_deg = np.degrees(np.arccos(np.clip(cos_val, -1, 1)))

fig = go.Figure()

# Draw vectors as arrows from origin
for vec, name, color in [(a, "a", "#636EFA"), (b, "b", "#EF553B")]:
    fig.add_trace(go.Scatter(
        x=[0, vec[0].item()], y=[0, vec[1].item()],
        mode="lines+markers",
        name=name,
        line=dict(color=color, width=3),
        marker=dict(size=[0, 10], symbol=["circle", "arrow"],
                    angleref="previous"),
    ))

# Draw projection
fig.add_trace(go.Scatter(
    x=[0, proj_vector[0].item()], y=[0, proj_vector[1].item()],
    mode="lines",
    name=f"proj(a onto b) = {proj_scalar.item():.2f}",
    line=dict(color="green", width=3, dash="dash"),
))

# Dashed line from a to its projection (the "drop" perpendicular)
fig.add_trace(go.Scatter(
    x=[a[0].item(), proj_vector[0].item()],
    y=[a[1].item(), proj_vector[1].item()],
    mode="lines",
    name="perpendicular",
    line=dict(color="gray", width=1, dash="dot"),
))

fig.update_layout(
    title=f"Dot product: a.b = {dot_val:.2f}, cos(angle) = {cos_val:.2f}, angle = {angle_deg:.1f} deg",
    xaxis=dict(scaleanchor="y", range=[-0.5, 4]),
    yaxis=dict(range=[-0.5, 3]),
    template="plotly_white",
    height=450,
    width=600,
)
fig.show()

Now in 3D, where we can see how the angle between vectors varies.

In [ ]:
section_header("3D Visualization", "Multiple vectors, multiple angles")

# Create several 3D vectors with known relationships
v1 = torch.tensor([1.0, 0.0, 0.0])
v2 = torch.tensor([0.9, 0.4, 0.1])  # close to v1
v3 = torch.tensor([0.0, 1.0, 0.0])  # orthogonal to v1
v4 = torch.tensor([-0.8, 0.2, 0.5]) # mostly opposite to v1

vectors_3d = {"v1": v1, "v2 (similar)": v2, "v3 (orthogonal)": v3, "v4 (opposite-ish)": v4}
colors = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA"]

fig = go.Figure()
for (name, v), color in zip(vectors_3d.items(), colors):
    v_np = v.numpy()
    cos = cosine_sim(v1, v).item()
    label = f"{name} (cos={cos:.2f})" if name != "v1" else name
    fig.add_trace(go.Scatter3d(
        x=[0, v_np[0]], y=[0, v_np[1]], z=[0, v_np[2]],
        mode="lines+markers",
        name=label,
        line=dict(color=color, width=5),
        marker=dict(size=[0, 5]),
    ))

fig.update_layout(
    title="3D Vectors: cosine similarity relative to v1",
    scene=dict(
        xaxis_title="x", yaxis_title="y", zaxis_title="z",
        aspectmode="cube",
    ),
    template="plotly_white",
    height=500,
    width=650,
)
fig.show()

**Key takeaway**: The dot product (and cosine similarity) encode the *geometric relationship* between vectors -- their angle. When we use embeddings, we rely on these relationships to find nearest neighbors, rank search results, or measure semantic similarity.

Now the question: what happens to these relationships when we compress the vectors?

---

## 3. Why Compression Is Hard: It's About Relationships, Not Values

Here is the core insight of this module. Suppose you have 100 embedding vectors and you want to compress them. The naive approach: round each floating-point value to fewer bits.

Rounding a value from 0.7823 to 0.75 seems harmless -- a tiny error. But the problem is not the error on individual values. The problem is what happens to the **pairwise relationships** between vectors.

Let's build this experiment step by step.

In [ ]:
section_header("The Quantization Experiment", "Round values, break relationships")

# Step 1: Create 100 random "embedding" vectors in 128 dimensions
n_vectors = 100
dim = 128
torch.manual_seed(42)
vectors = torch.randn(n_vectors, dim)

# Normalize them (common for embeddings)
vectors = vectors / vectors.norm(dim=1, keepdim=True)

print(f"Created {n_vectors} unit vectors in {dim} dimensions")
print(f"Shape: {vectors.shape}")
print(f"Norm of first vector: {vectors[0].norm().item():.4f} (should be ~1.0)")

In [ ]:
# Step 2: Compute all pairwise dot products -- the ground truth
# For unit vectors, dot product = cosine similarity
true_dots = vectors @ vectors.T  # shape: (100, 100)

print(f"Pairwise dot product matrix shape: {true_dots.shape}")
print(f"Diagonal values (self-similarity): {true_dots.diag()[:5]}")
print(f"  (should all be ~1.0 since vectors are unit length)")
print(f"\nOff-diagonal range: [{true_dots[~torch.eye(n_vectors, dtype=bool)].min():.4f}, "
      f"{true_dots[~torch.eye(n_vectors, dtype=bool)].max():.4f}]")

In [ ]:
# Step 3: Naive quantization -- round values to a grid with `bits` bits of precision

def naive_quantize(x, bits):
    """Round each value to one of 2^bits uniformly spaced levels.
    
    Maps the range [min, max] of x to 2^bits discrete levels,
    then reconstructs (dequantizes) back to float.
    """
    n_levels = 2 ** bits
    x_min, x_max = x.min(), x.max()
    # Scale to [0, n_levels - 1], round to integers, scale back
    scale = (x_max - x_min) / (n_levels - 1)
    codes = torch.round((x - x_min) / scale)  # integer codes
    x_hat = codes * scale + x_min               # reconstructed values
    return x_hat

# Example: quantize to 4 bits (16 levels)
vectors_q4 = naive_quantize(vectors, bits=4)

print("Original first 5 values:", vectors[0, :5].tolist())
print("Quantized first 5 values:", vectors_q4[0, :5].tolist())
print(f"\nMax per-element error: {(vectors - vectors_q4).abs().max().item():.6f}")
print("Looks small, right? Let's see what it does to relationships...")

In [ ]:
# Step 4: Recompute pairwise dot products after quantization
quantized_dots_4bit = vectors_q4 @ vectors_q4.T

# Compare: how much did the relationships change?
# Extract upper triangle (excluding diagonal) for comparison
mask = torch.triu(torch.ones(n_vectors, n_vectors, dtype=bool), diagonal=1)
true_pairs = true_dots[mask]
q4_pairs = quantized_dots_4bit[mask]

correlation = torch.corrcoef(torch.stack([true_pairs, q4_pairs]))[0, 1].item()
print(f"Correlation between true and 4-bit dot products: {correlation:.4f}")
print(f"Mean absolute error in dot products: {(true_pairs - q4_pairs).abs().mean().item():.4f}")

### Does nearest-neighbor search survive?

The real test: if you ask "which are the 10 most similar vectors to vector $i$?", do you get the same answer before and after quantization?

In [ ]:
def recall_at_k(true_dots, approx_dots, k=10):
    """What fraction of the true top-k neighbors appear in the approximate top-k?
    
    Averaged over all query vectors.
    """
    n = true_dots.shape[0]
    recalls = []
    for i in range(n):
        # Exclude self (diagonal)
        true_row = true_dots[i].clone()
        approx_row = approx_dots[i].clone()
        true_row[i] = -float("inf")
        approx_row[i] = -float("inf")
        
        true_topk = set(torch.topk(true_row, k).indices.tolist())
        approx_topk = set(torch.topk(approx_row, k).indices.tolist())
        recalls.append(len(true_topk & approx_topk) / k)
    return np.mean(recalls)

r_at_10 = recall_at_k(true_dots, quantized_dots_4bit, k=10)
print(f"Recall@10 with 4-bit quantization: {r_at_10:.2%}")
print(f"  (100% means perfect -- every true neighbor is found)")

That might not look terrible yet. Let's see what happens as we use fewer bits.

---

## 4. Visualize the Damage

Now we sweep across bit widths and watch the geometry degrade.

In [ ]:
section_header("Sweeping Bit Widths", "Watch the geometry degrade")

bit_widths = [8, 6, 4, 3, 2, 1]
results = {}

for bits in bit_widths:
    vq = naive_quantize(vectors, bits)
    dq = vq @ vq.T
    
    corr = torch.corrcoef(torch.stack([true_dots[mask], dq[mask]]))[0, 1].item()
    mae = (true_dots[mask] - dq[mask]).abs().mean().item()
    r10 = recall_at_k(true_dots, dq, k=10)
    
    results[bits] = {"correlation": corr, "mae": mae, "recall@10": r10}
    print(f"{bits}-bit:  correlation={corr:.4f}  MAE={mae:.4f}  recall@10={r10:.2%}")

In [ ]:
# Scatter plot: true vs quantized dot products for several bit widths
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[f"{b}-bit (recall@10={results[b]['recall@10']:.0%})" for b in bit_widths],
)

for idx, bits in enumerate(bit_widths):
    row = idx // 3 + 1
    col = idx % 3 + 1
    vq = naive_quantize(vectors, bits)
    dq = vq @ vq.T
    
    # Sample 500 pairs for readability
    indices = torch.randperm(true_pairs.shape[0])[:500]
    fig.add_trace(
        go.Scattergl(
            x=true_pairs[indices].numpy(),
            y=dq[mask][indices].numpy(),
            mode="markers",
            marker=dict(size=2, opacity=0.5),
            showlegend=False,
        ),
        row=row, col=col,
    )
    # Perfect line
    fig.add_trace(
        go.Scatter(
            x=[-0.3, 0.4], y=[-0.3, 0.4],
            mode="lines",
            line=dict(color="red", dash="dash", width=1),
            showlegend=False,
        ),
        row=row, col=col,
    )

fig.update_layout(
    title="True vs. Quantized Dot Products (each point = one vector pair)",
    template="plotly_white",
    height=600,
    width=900,
)
fig.update_xaxes(title_text="True dot product", row=2)
fig.update_yaxes(title_text="Quantized dot product", col=1)
fig.show()

In [ ]:
# Recall@10 vs bit width
bits_list = sorted(results.keys())
recalls = [results[b]["recall@10"] for b in bits_list]
correlations = [results[b]["correlation"] for b in bits_list]

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=bits_list, y=recalls,
        mode="lines+markers",
        name="Recall@10",
        line=dict(color="#636EFA", width=3),
        marker=dict(size=10),
    ),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=bits_list, y=correlations,
        mode="lines+markers",
        name="Dot Product Correlation",
        line=dict(color="#EF553B", width=3),
        marker=dict(size=10),
    ),
    secondary_y=True,
)

fig.update_layout(
    title="How Naive Quantization Destroys Geometry",
    xaxis_title="Bits per value",
    template="plotly_white",
    height=450,
    width=700,
)
fig.update_yaxes(title_text="Recall@10", secondary_y=False, range=[0, 1.05])
fig.update_yaxes(title_text="Correlation", secondary_y=True, range=[0, 1.05])
fig.show()

### What just happened?

Even at 4 bits per value (16 levels), naive rounding introduces enough noise to scramble the neighbor rankings. At 2 bits (4 levels), nearest-neighbor search is nearly useless.

The per-element errors are small. But errors **accumulate** across 128 dimensions when you compute dot products, and the accumulated error is large enough to reorder which vectors appear closest.

This is the fundamental challenge:

> **We need compression that preserves the GEOMETRY (distances, angles, inner products), not just the individual values.**

---

## 5. The Challenge Framed

Here is where we stand:

1. Embeddings encode meaning as vectors.
2. Similarity = dot product = angle between vectors.
3. Naive quantization introduces small per-element errors that compound into large relationship errors.
4. Nearest-neighbor search breaks well before the individual values look bad.

We need a smarter approach. Specifically, we need quantization that:
- Uses far fewer bits than float32 (memory savings)
- **Preserves pairwise inner products** as accurately as possible
- Ideally gives **unbiased** estimates of the true inner product

This is exactly what **TurboQuant** achieves. Over the next modules, we will build every piece of it from scratch.

---

## Exercise: Ternary Quantization

Here is an extreme case. Suppose we quantize every value to one of just three levels: **{-1, 0, 1}** (ternary quantization). Each value gets mapped to the nearest of these three.

**Before running the experiment, make your prediction**: will recall@10 be above or below 50%?

Think about it. We have 128 dimensions but only 3 possible values per dimension. That is $3^{128}$ possible distinct vectors -- astronomically many. But each coordinate is very coarsely represented.

In [ ]:
# YOUR PREDICTION: Will recall@10 be above or below 50%?
# Set this to True if you think recall@10 > 0.5, False otherwise.
my_prediction_above_50 = True  # <-- change this to your prediction

In [ ]:
# Ternary quantization: map each value to {-1, 0, 1}
def ternary_quantize(x):
    """Map each value to the nearest of {-1, 0, 1}."""
    return torch.round(x).clamp(-1, 1)

vectors_ternary = ternary_quantize(vectors)
ternary_dots = vectors_ternary @ vectors_ternary.T
ternary_recall = recall_at_k(true_dots, ternary_dots, k=10)

print(f"Ternary quantization recall@10: {ternary_recall:.2%}")
print(f"Your prediction (above 50%): {my_prediction_above_50}")
print(f"Actual result above 50%: {ternary_recall > 0.5}")
print()

# Check with the helper
check_intuition(
    result=ternary_recall > 0.5,
    expected=my_prediction_above_50,
    your_input=f"predicted {'above' if my_prediction_above_50 else 'below'} 50%",
    hint=(
        "Unit vectors in 128 dims have values roughly in [-0.2, 0.2]. "
        "Rounding to {-1, 0, 1} maps almost everything to 0, destroying most information."
    ),
    label="Prediction matches reality",
)

In [ ]:
# Let's understand WHY by looking at the distribution of values
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=vectors.flatten().numpy(),
    nbinsx=80,
    name="Original values",
    opacity=0.7,
))

fig.add_trace(go.Histogram(
    x=vectors_ternary.flatten().numpy(),
    nbinsx=20,
    name="Ternary values",
    opacity=0.7,
))

fig.update_layout(
    title="Value distributions: original vs ternary",
    xaxis_title="Value",
    yaxis_title="Count",
    barmode="overlay",
    template="plotly_white",
    height=400,
    width=700,
)
fig.show()

# What fraction of values mapped to each level?
for level in [-1, 0, 1]:
    frac = (vectors_ternary == level).float().mean().item()
    print(f"Fraction mapped to {level:+d}: {frac:.2%}")

print("\nFor unit vectors in 128-d, values are typically in [-0.2, 0.2].")
print("Almost everything rounds to 0 -- the information is destroyed.")

### Why ternary fails so badly

Unit vectors in 128 dimensions have coordinates distributed roughly as $\mathcal{N}(0, 1/128)$, so most values are between -0.2 and 0.2. Rounding to {-1, 0, 1} maps almost everything to 0.

The lesson: naive quantization does not account for the *distribution* of the data. The grid levels {-1, 0, 1} are a terrible match for values concentrated near zero.

In Module 02, we will build the **Lloyd-Max** algorithm, which places quantization levels optimally for a given distribution. And in Module 04, we will see how **random rotations** reshape the distribution to make even uniform quantization work well.

---

## Summary

- **Dot products** measure the geometric relationship between vectors (angle, projection).
- **Embeddings** rely on these relationships for search and similarity.
- **Naive quantization** (rounding to fewer bits) introduces small per-element errors that **compound** across dimensions.
- **Nearest-neighbor search breaks** well before individual values look bad.
- We need compression that preserves **geometry**, not just values. This is the problem TurboQuant solves.

**Next**: Module 02 -- Scalar Quantization & Lloyd-Max. The first building block.